In [1]:
import numpy as np
import pandas as pd

In [20]:
movies=pd.read_csv('tmdb_5000_movies.csv')
credits=pd.read_csv('tmdb_5000_credits.csv')

In [21]:
movies.head(1) # to see the dataframe distribution

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [22]:
credits.head(1)['cast'] #to see the dataframe distribution
# credits.head(1)['cast'].values, to see the column values

0    [{"cast_id": 242, "character": "Jake Sully", "...
Name: cast, dtype: object

In [23]:
movies.shape

(4803, 20)

In [24]:
credits.shape

(4803, 4)

In [25]:
# as we have two different dataframes, so we decide to merge them (on the basis of id/titlle we can do it)
movies = movies.merge(credits, on='title')

so total is 20+4-1=23 columns.

In [27]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


Thus above we added the two dataframes, this would be helpful as we had two different data frames initiallly. Now, our aim would be to remove all those columns which are not usefull to us.

In [30]:
#movies['original_language'].value_counts(), mostly its 99% english so won't be that helpful

In [34]:
# List of columns we choose to keep (we keep those columns which help us to keep tags in content based generator system)
# genres
# id (for posters in website)
# keywords
# title
# overview
# cast
# crew
movies=movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [37]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


(overview+genre+keywords+cast+crew) is merged to make tags, in total we would have movie_id, title, tags.

cast- we choose only the top-3 casts
crew- we take only the directors
we add all these in overview, paragraphs

In [38]:
# now we handle the missing and duplicate data
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [40]:
movies.dropna(inplace=True)

C:\Users\HP\AppData\Local\Temp\ipykernel_4396\3786870272.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies.dropna(inplace=True)


In [41]:
movies.isnull().sum() # as 3 wasn't a big number so we decide to drop it 

movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [42]:
# duplicate data
movies.duplicated().sum()

0

In [43]:
movies.iloc[0].genres #to see the format of genre

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

We want it in this form, ['action',....], but it's in a weird format where we have dictionaries inside of lists

In [57]:
import ast

In [58]:
from ast import literal_eval

In [59]:
def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L
# but it's a string of list and not list, so we sue ast model, literal_eval 

In [60]:
movies['genres']= movies['genres'].apply(convert)

In [61]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [62]:
# next we will do the same for keywuords
movies['keywords']= movies['keywords'].apply(convert)

In [65]:
# in cast, we need first three names
def convert3(obj):
    L=[]
    counter=0
    for i in ast.literal_eval(obj):
        if counter!=3:
            L.append(i['name'])
        else:
            break
           
    return L

In [66]:
movies['cast']= movies['cast'].apply(convert3)

In [67]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [68]:
movies['crew'][0] # we want its job's value is director

'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"},

In [69]:
def fetch_director(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L

In [70]:
movies['crew']= movies['crew'].apply(fetch_director)

In [71]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...",[James Cameron]


In [73]:
# now we intend to convert overview, which is a string to convert into lists, so that we can concardinate
movies['overview']=movies['overview'].apply(lambda x : x.split())

In [74]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley, ...",[Gore Verbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux, R...",[Sam Mendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman, A...",[Christopher Nolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton,...",[Andrew Stanton]


In [75]:
# we have to apply some transformation so that there is no space in b/w names
# tags of Aman Behera, and Aman Mehera would have 'Aman' in common so tag amy cause confussion
# so we wish to remove the gaps
movies['genres'] = movies['genres'].apply(lambda x : [i.replace(" "," ") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x : [i.replace(" "," ") for i in x])
movies['cast'] = movies['cast'].apply(lambda x : [i.replace(" "," ") for i in x])
movies['crew'] = movies['crew'].apply(lambda x : [i.replace(" "," ") for i in x])

In [76]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley, ...",[Gore Verbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux, R...",[Sam Mendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman, A...",[Christopher Nolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton,...",[Andrew Stanton]


In [77]:
# now we concardinate all the 4 columns in to a new column named as tags
movies['tags']=movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [80]:
# we form a new dataframe where we only keep the necessary data frames
new_df=movies[['movie_id','title','tags']]

In [83]:
new_df['tags']=new_df['tags'].apply(lambda x : " ".join(x)) # we intend to join the list, where there is a space

C:\Users\HP\AppData\Local\Temp\ipykernel_4396\2688907974.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda x : " ".join(x)) # we intend to join the list, where there is a space


In [84]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [85]:
new_df['tags'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy Science Fiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d Sam Worthington Zoe Saldana Sigourney Weaver Stephen Lang Michelle Rodriguez Giovanni Ribisi Joel David Moore CCH Pounder Wes Studi Laz Alonso Dileep Rao Matt Gerald Sean Anthony Moran Jason Whyte Scott Lawrence Kelly Kilgour James Patrick Pitt Sean Patrick Murphy Peter Dillon Kevin Dorman Kelson Henderson David Van Horn Jacob Tomuri Michael Blain-Rozgay Jon Curry Luke Hawker Woody Schultz Peter Mensah Sonia Yee Jahnel Curfman Ilram Choi Kyla Warren Lisa Roumain Debra Wilson Chris Mala Taylor Kibby Jodie Landau Julie Lamm Cullen B. Madden Joseph Brady Madden Frankie Torres Austin 

In [86]:
# now we convert everything into lower case
new_df['tags']=new_df['tags'].apply(lambda x :x.lower())

C:\Users\HP\AppData\Local\Temp\ipykernel_4396\2773367542.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda x :x.lower())


In [87]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


text vectorisation
as we want to find the simialrity b/w two tags, as its all textual data so we need to convert it to numbers to quantify the similarity

we would be using bag of words

here we combine all the tags of the words, and we use the most frequent words, like say 5k most used words. and then go again
to each tag and check how many times these words occur in each tag, and use this dataframe of 5k x 5k and each row would be a vector

we also remove the stop words like a,are,the,its etc asd they help in sentence formation

In [99]:
from sklearn.feature_extraction.text import CountVectorizer
cv= CountVectorizer(max_features=5000, stop_words='english') #instance created

In [112]:
vectors=cv.fit_transform(new_df['tags']).toarray() # and convert it into numpy array

In [113]:
vectors # vectorisation done, here there would a lot of zeros as it would be sparse.

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [114]:
cv.get_feature_names_out()

array(['000', '10', '11', ..., 'zooey', 'zoë', 'zucker'], dtype=object)

action/action is kinda similar, similarily more words would be there which have same contextual meaning.

So we apply STEMING, like loving, loved and love would be converted to love, so we use NLTK

In [104]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: nltk in c:\users\hp\anaconda3\lib\site-packages (3.8.1)



In [105]:
import nltk

In [106]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [108]:
# example 
ps.stem('loved')

'love'

In [109]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
        
    return " ".join(y)

In [110]:
new_df['tags']=new_df['tags'].apply(stem)

C:\Users\HP\AppData\Local\Temp\ipykernel_4396\3514595201.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(stem)


In [111]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."


we have to calculate the distance of each movie with every other movie in the 5000 dimensional space
we would find the cosine distance and not the euclid distance.
check on curse of dimensionality as in higher dimensions euclid distance

In [115]:
from sklearn.metrics.pairwise import cosine_similarity

In [120]:
# pass the vectors into cosine simialrity
similarity=cosine_similarity(vectors)

Logical as we find the distance of evry movie with every other movie so its n^2 i.e. (4806 x 4806)

In [121]:
similarity[0]

array([1.        , 0.10881351, 0.07927124, ..., 0.03464015, 0.01900543,
       0.01536191])

Tells us the similarity of 1st moview with every other movie, hence of this matrix the diagonals would be 1.

In [125]:
sorted(similarity[0], reverse=True)

[1.0000000000000002,
 0.3017318023494271,
 0.30149839166852865,
 0.281451220037393,
 0.26532607018881077,
 0.2623611139382941,
 0.25966362889046096,
 0.253825649699079,
 0.2490384025487984,
 0.24790171554530396,
 0.2466984290363284,
 0.2401888567236763,
 0.2389433395128927,
 0.23624607804052644,
 0.23481812262849727,
 0.2337367852823266,
 0.23348563864528488,
 0.23268303344655972,
 0.23211322981563567,
 0.2313248221920752,
 0.2290086764844313,
 0.22893627067137456,
 0.22890761423482825,
 0.22811268756929876,
 0.22806517893865344,
 0.2277196711990031,
 0.22595384385027567,
 0.22422253919312196,
 0.22212349222678546,
 0.22177267137135853,
 0.21998299271308816,
 0.21773457205625582,
 0.21694136514992582,
 0.21567839629547517,
 0.2150490055616034,
 0.21490757246142353,
 0.21394759496263566,
 0.2125285372954286,
 0.21225525397479206,
 0.21212175169479097,
 0.21166219538630943,
 0.2104141762796303,
 0.21035158095583564,
 0.20979072549805627,
 0.208756546856263,
 0.20819808786086785,
 0.20773

But if we do sorting, then its index position would be changed, so we enumerate it

In [130]:
sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x : x[1])[1:6] # we say that we want to sort on the basis of second

[(47, 0.3017318023494271),
 (2409, 0.30149839166852865),
 (149, 0.281451220037393),
 (65, 0.26532607018881077),
 (1216, 0.2623611139382941)]

In [138]:
# now we make a function,if we say a movie it would tell us 5 movies whose similarity with the original movie is the highest
# first we find the index of that movie 
# second we would sort the array of similarity of that index,a and select the first 6 places , excluding itself.
def recommend(movie):
    movie_index= new_df[new_df['title']==movie].index[0]
    distances=similarity[movie_index]
    movies_list=sorted(list(enumerate(distances)), reverse=True, key=lambda x : x[1])[1:6]
    
    for i in movies_list:
        print(new_df.iloc[i[0]].title)
    
    return

In [140]:
recommend('Batman Begins')

The Dark Knight Rises
The Dark Knight
Superman
The Fifth Element
The Departed
